# Sprint 4 — 13_ensembles.ipynb
## Rol: Ensemble Engineer (PB-14)

### Objetivo
Construir ensambles y modelos avanzados (Voting, Stacking, XGBoost, LightGBM), evaluarlos con CV y compararlos contra los tuned individuales del Sprint 4.

### Decisiones de este template
- **Top 3 por defecto**: `rf`, `gb`, `knn`, pero puedes cambiarlos fácilmente.
- Este notebook intenta trabajar con los modelos tuneados del rol anterior.
- **No usa el test set final**.
- El criterio principal por defecto sigue siendo **precision**.

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
PROJECT_ROOT = Path(".").resolve().parent if Path.cwd().name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

MODELS_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "figures" / "sprint4"

TARGET_COL = "IsBadBuy"
PRIMARY_METRIC = "precision"

USE_TUNING_FILE = True
DEFAULT_TOP_CANDIDATES = ["rf", "gb", "knn"]
TOP_K = 3
ENSEMBLE_DATA_VARIANT = "general"

TUNING_RESULTS_PATH = MODELS_DIR / "tuning_results.csv"
MODEL_SELECTION_PATH = MODELS_DIR / "model_selection_candidates.csv"
TRAIN_GENERAL_PATH = DATA_DIR / "train_balanced.csv"
TRAIN_KNN_PATH = DATA_DIR / "train_balanced_knn_raw.csv"
PREPROCESSOR_GENERAL_PATH = MODELS_DIR / "preprocessor.pkl"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
train_general_df = pd.read_csv(TRAIN_GENERAL_PATH)
train_knn_df = pd.read_csv(TRAIN_KNN_PATH) if TRAIN_KNN_PATH.exists() else train_general_df.copy()

datasets = {"general": train_general_df, "knn_raw": train_knn_df}

def get_xy(data_variant: str):
    df = datasets[data_variant].copy()
    X = df.drop(columns=[TARGET_COL]).copy()
    y = df[TARGET_COL].copy()
    return X, y

X_ensemble, y_ensemble = get_xy(ENSEMBLE_DATA_VARIANT)

In [ ]:
if USE_TUNING_FILE and TUNING_RESULTS_PATH.exists():
    tuning_results_df = pd.read_csv(TUNING_RESULTS_PATH)
    active_candidates = tuning_results_df["model_key"].dropna().astype(str).tolist()
elif MODEL_SELECTION_PATH.exists():
    selected_df = pd.read_csv(MODEL_SELECTION_PATH)
    active_candidates = selected_df["model_key"].dropna().astype(str).tolist()
else:
    active_candidates = DEFAULT_TOP_CANDIDATES.copy()

active_candidates = active_candidates[:TOP_K] if TOP_K is not None else active_candidates
print("Candidatos activos:", active_candidates)

In [ ]:
tuned_model_paths = {m: MODELS_DIR / f"tuned_{m}.pkl" for m in active_candidates}
tuned_models = {m: joblib.load(p) for m, p in tuned_model_paths.items() if p.exists()}
print("Modelos tuneados cargados:", list(tuned_models.keys()))

In [ ]:
compatibility_rows = []
compatible_models = {}
X_sample = X_ensemble.head(5).copy()

for model_key, model in tuned_models.items():
    try:
        _ = model.predict(X_sample)
        compatible = True
        compatible_models[model_key] = model
        err = ""
    except Exception as e:
        compatible = False
        err = str(e)

    compatibility_rows.append({
        "model_key": model_key,
        "compatible_with_ensemble_X": compatible,
        "data_variant_tested": ENSEMBLE_DATA_VARIANT,
        "notes": err[:200],
    })

compatibility_df = pd.DataFrame(compatibility_rows)
display(compatibility_df)

In [ ]:
xgb_available = False
lgbm_available = False

try:
    from xgboost import XGBClassifier
    xgb_available = True
except Exception:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
    lgbm_available = True
except Exception:
    LGBMClassifier = None

print("XGBoost available:", xgb_available)
print("LightGBM available:", lgbm_available)

In [ ]:
def build_pipeline_from_preprocessor_artifact(clf, artifact_path=PREPROCESSOR_GENERAL_PATH):
    obj = joblib.load(artifact_path)
    steps = []
    if isinstance(obj, dict):
        if "feature_builder" in obj:
            steps.append(("feature_builder", obj["feature_builder"]))
        if "preprocessor" in obj:
            steps.append(("preprocessor", obj["preprocessor"]))
    else:
        steps.append(("preprocessor", obj))
    steps.append(("clf", clf))
    return Pipeline(steps)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1", "roc_auc": "roc_auc"}

In [ ]:
ensemble_results_rows = []

if len(compatible_models) >= 2:
    voting_soft = VotingClassifier(estimators=[(k, v) for k, v in compatible_models.items()], voting="soft")
    scores = cross_validate(voting_soft, X_ensemble, y_ensemble, cv=cv, scoring=scoring, n_jobs=-1)
    voting_soft.fit(X_ensemble, y_ensemble)
    voting_path = MODELS_DIR / "ensemble_voting_soft.pkl"
    joblib.dump(voting_soft, voting_path)

    ensemble_results_rows.append({
        "model_key": "voting_soft",
        "candidate_type": "ensemble",
        "data_variant": ENSEMBLE_DATA_VARIANT,
        "artifact_path": str(voting_path),
        "test_accuracy_mean": np.mean(scores["test_accuracy"]),
        "test_precision_mean": np.mean(scores["test_precision"]),
        "test_recall_mean": np.mean(scores["test_recall"]),
        "test_f1_mean": np.mean(scores["test_f1"]),
        "test_roc_auc_mean": np.mean(scores["test_roc_auc"]),
        "fit_time_mean": np.mean(scores["fit_time"]),
        "score_time_mean": np.mean(scores["score_time"]),
        "notes": f"Soft voting with {list(compatible_models.keys())}",
    })

    stacking = StackingClassifier(
        estimators=[(k, v) for k, v in compatible_models.items()],
        final_estimator=LogisticRegression(max_iter=1000),
        cv=5,
        n_jobs=-1,
    )
    scores = cross_validate(stacking, X_ensemble, y_ensemble, cv=cv, scoring=scoring, n_jobs=-1)
    stacking.fit(X_ensemble, y_ensemble)
    stacking_path = MODELS_DIR / "ensemble_stacking.pkl"
    joblib.dump(stacking, stacking_path)

    ensemble_results_rows.append({
        "model_key": "stacking",
        "candidate_type": "ensemble",
        "data_variant": ENSEMBLE_DATA_VARIANT,
        "artifact_path": str(stacking_path),
        "test_accuracy_mean": np.mean(scores["test_accuracy"]),
        "test_precision_mean": np.mean(scores["test_precision"]),
        "test_recall_mean": np.mean(scores["test_recall"]),
        "test_f1_mean": np.mean(scores["test_f1"]),
        "test_roc_auc_mean": np.mean(scores["test_roc_auc"]),
        "fit_time_mean": np.mean(scores["fit_time"]),
        "score_time_mean": np.mean(scores["score_time"]),
        "notes": f"Stacking with {list(compatible_models.keys())}",
    })

if xgb_available:
    xgb_pipe = build_pipeline_from_preprocessor_artifact(
        XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric="logloss",
            use_label_encoder=False,
        )
    )
    Xg, yg = get_xy("general")
    scores = cross_validate(xgb_pipe, Xg, yg, cv=cv, scoring=scoring, n_jobs=-1)
    xgb_pipe.fit(Xg, yg)
    xgb_path = MODELS_DIR / "xgb_model.pkl"
    joblib.dump(xgb_pipe, xgb_path)

    ensemble_results_rows.append({
        "model_key": "xgb",
        "candidate_type": "advanced_boosting",
        "data_variant": "general",
        "artifact_path": str(xgb_path),
        "test_accuracy_mean": np.mean(scores["test_accuracy"]),
        "test_precision_mean": np.mean(scores["test_precision"]),
        "test_recall_mean": np.mean(scores["test_recall"]),
        "test_f1_mean": np.mean(scores["test_f1"]),
        "test_roc_auc_mean": np.mean(scores["test_roc_auc"]),
        "fit_time_mean": np.mean(scores["fit_time"]),
        "score_time_mean": np.mean(scores["score_time"]),
        "notes": "XGBoost pipeline",
    })

if lgbm_available:
    lgbm_pipe = build_pipeline_from_preprocessor_artifact(
        LGBMClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbose=-1,
        )
    )
    Xg, yg = get_xy("general")
    scores = cross_validate(lgbm_pipe, Xg, yg, cv=cv, scoring=scoring, n_jobs=-1)
    lgbm_pipe.fit(Xg, yg)
    lgbm_path = MODELS_DIR / "lgbm_model.pkl"
    joblib.dump(lgbm_pipe, lgbm_path)

    ensemble_results_rows.append({
        "model_key": "lgbm",
        "candidate_type": "advanced_boosting",
        "data_variant": "general",
        "artifact_path": str(lgbm_path),
        "test_accuracy_mean": np.mean(scores["test_accuracy"]),
        "test_precision_mean": np.mean(scores["test_precision"]),
        "test_recall_mean": np.mean(scores["test_recall"]),
        "test_f1_mean": np.mean(scores["test_f1"]),
        "test_roc_auc_mean": np.mean(scores["test_roc_auc"]),
        "fit_time_mean": np.mean(scores["fit_time"]),
        "score_time_mean": np.mean(scores["score_time"]),
        "notes": "LightGBM pipeline",
    })

ensemble_results_df = pd.DataFrame(ensemble_results_rows).sort_values(by="test_precision_mean", ascending=False).reset_index(drop=True)
display(ensemble_results_df)

In [ ]:
ensemble_vs_tuned_df = ensemble_results_df[["model_key", "test_precision_mean"]].rename(columns={"test_precision_mean": "ensemble_precision_cv"}) if not ensemble_results_df.empty else pd.DataFrame()
display(ensemble_vs_tuned_df)

In [ ]:
ENSEMBLE_RESULTS_PATH = MODELS_DIR / "ensemble_results.csv"
ENSEMBLE_VS_TUNED_PATH = MODELS_DIR / "ensemble_vs_tuned.csv"

ensemble_results_df.to_csv(ENSEMBLE_RESULTS_PATH, index=False)
if not ensemble_vs_tuned_df.empty:
    ensemble_vs_tuned_df.to_csv(ENSEMBLE_VS_TUNED_PATH, index=False)

print("Saved:", ENSEMBLE_RESULTS_PATH)
print("Saved:", ENSEMBLE_VS_TUNED_PATH if not ensemble_vs_tuned_df.empty else "comparison skipped")

In [ ]:
if not ensemble_results_df.empty:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(ensemble_results_df["model_key"], ensemble_results_df["test_precision_mean"])
    ax.set_title("Ensemble / Advanced Models — Precision (CV)")
    ax.set_ylabel("Precision")
    plt.tight_layout()
    ensemble_fig_path = REPORTS_DIR / "ensemble_comparison.png"
    plt.savefig(ensemble_fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", ensemble_fig_path)